In [ ]:
from fastwarc.warc import ArchiveIterator, WarcRecordType
import resiliparse.extract.html2text
from typing import Literal
import rich

WARC_FILE = "../local-shared-data/CC/example.warc.gz"
WET_FILE = "../local-shared-data/CC/example.warc.wet.gz"

In [ ]:
resiliparse.extract.html2text.extract_plain_text("<html><body>Hello, world!</body></html>")

In [ ]:
3+5

## Code to peak through CC entries and get entries by URI (to compare WARC to WET)

In [ ]:
## Retrieve a record by index or by URI 

def get_record_uri(uri, mode: Literal["wet", "warc"]):
    fname = WARC_FILE if mode == "warc" else WET_FILE
    with open(fname, "rb") as f:
        it = ArchiveIterator(f)
        val = next(it)
        while True: 
            val = next(it)
            if mode == "warc" and val.headers.get("Content-Type", "NONE") != "application/http; msgtype=response": continue
            if val.headers.get('WARC-Target-URI', "NONE") == uri: break
        return val

def get_record_ind(ind, mode: Literal["wet", "warc"]):
    fname = WARC_FILE if mode == "warc" else WET_FILE
    with open(fname, "rb") as f:
        it = ArchiveIterator(f)
        val = next(it)
        i = 0
        while i < ind:
            val = next(it)
            if mode == "warc":
                # i += 1
                if val.headers.get("Content-Type", "NONE") == "application/http; msgtype=response": i += 1
            else:
                i += 1
        return val

In [ ]:
## Print some info about some retrieved records
for i in range(50, 100):
    wet_rec = get_record_ind(i, "wet")
    warc_rec = get_record_ind(i, "warc")

    print(i, warc_rec.headers.get('WARC-Target-URI', "NONE"), wet_rec.headers.get('WARC-Target-URI', "NONE"))
    print(rich.print(wet_rec.reader.read().decode()[:15]), 'len =', wet_rec.headers['Content-Length'])

In [ ]:
## Compare WARC and WET text for a fixed URI

ind = 60
uri = "http://au-urlm.com/www.arrse.co.uk"

wet_rec = get_record_uri(uri, "wet")
warc_rec = get_record_uri(uri, "warc")

# wet_rec = get_record_ind(ind, "wet")
# warc_rec = get_record_ind(ind, "warc")

rich.print(dict(warc_rec.headers))

warc_str = warc_rec.reader.read().decode()
warc_txt = resiliparse.extract.html2text.extract_plain_text(warc_str)
wet_txt = wet_rec.reader.read().decode()
print(warc_rec.headers['WARC-Target-URI'])
print(wet_rec.headers['WARC-Target-URI'])

print("WARC full:")
print(warc_str)


print("WARC text:")
print(warc_txt)

print("\n\nWET text:")
print(wet_txt)


In [ ]:
## Test to see how many WARC files there are. There are ~60k.

with open(WARC_FILE, "rb") as f:
    it = ArchiveIterator(f)
    for i, entry in enumerate(it):
        if i % 1000 == 0:
            print(i)

## Make an iterator to run over WARC files and text extracted w resiliparse

In [ ]:
## Make an iterator over WARC entries, returning the entry along with extracted plain text using resiliparse
## Use as for entry, parsed_text in warc_text_iterator()
def warc_text_iterator(): 
    with open(WARC_FILE, "rb") as f:
        it = ArchiveIterator(f)
        for entry in it:
            if entry.headers.get("Content-Type", "NONE") != "application/http; msgtype=response": continue
            try: 
                warc_str = entry.reader.read().decode()
                warc_txt = resiliparse.extract.html2text.extract_plain_text(warc_str).strip()
                if len(warc_txt) == 0: continue
                yield entry, warc_txt
            except UnicodeDecodeError:
                continue

## Code to test language identification

Uses the fastText lid.176.bin  classifier

In [ ]:
import fasttext
lang_detect_model = fasttext.load_model("../local-shared-data/classifiers/lid.176.bin")

In [ ]:
def get_lang(text):
    stripped = text.replace("\n", " ").strip()
    lan, conf = lang_detect_model.predict(stripped)
    lan = lan[0]
    conf = conf[0]
    return lan, conf


def is_english(text, english_cutoff = 0.80):
    lang, conf = get_lang(text)
    return lan == "__label__en" and conf >= english_cutoff

In [ ]:
## Iterate through first 1000 WARC files. Can print text if desired. 
upto = 1000
conf_range = [0.9, 1.0]
print("CONF RANGE:", conf_range)
full_str = ""
for i, (entry, text) in enumerate(warc_text_iterator()):
    if i > upto: break
    lan, conf = get_lang(text)
    engl = is_english(text)
    
    if lan == '__label__en' and conf >= conf_range[0] and conf <= conf_range[1]: 
        print(f"Ind: {i}, uri: {entry.headers.get('WARC-Target-URI', "NONE")}, length: {entry.headers['Content-Length']}, lan: {lan}, conf: {conf}")
        print(text)
        print("\n\n--------------------\n\n")
        full_str += f"Ind: {i}, uri: {entry.headers.get('WARC-Target-URI', "NONE")}, length: {entry.headers['Content-Length']}, lan: {lan}, conf: {conf}"
        full_str += text
        full_str += "\n\n--------------------\n\n"

fname = f"../notes/conf_ran_engl_{conf_range[0]}_{conf_range[1]}"
with open(fname, 'w') as f: 
    f.write(full_str)


## Mask personal information
Write functions to mask email address, phone numbers, IP address using regexp

should replace email with |||EMAIL_ADDRESS||, etc.

In [ ]:
import re

In [ ]:
EMAIL_REGEX = r"""[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"""
PHONE_REGEX = r"""^(?:\+?\d{1,3}[-. ]?)?\(?([0-9]{3})\)?[-. ]?([0-9]{3})[-. ]?([0-9]{4})$""" 
IP_REGEX = r"""(?:(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\.){3}(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)""" 


COMPILED_EMAIL_REGEX = re.compile(EMAIL_REGEX)
COMPILED_PHONE_REGEX = re.compile(PHONE_REGEX)
COMPILED_IP_REGEX = re.compile(IP_REGEX)

In [ ]:
re.findall(COMPILED_IP_REGEX, "192.168.1.1 234.123.53.23")
re.findall(COMPILED_PHONE_REGEX, "+1 343 535 3421 , 646-895-2843 , 123.433.5312")

In [ ]:
def mask_emails(str):
    return re.sub(COMPILED_EMAIL_REGEX, "|||EMAIL_ADDRESS|||", str)

def mask_phone(str):
    return re.sub(COMPILED_PHONE_REGEX, "|||PHONE_NUMBER|||", str)

def mask_ip(str):
    return re.sub(COMPILED_IP_REGEX, "|||IP_ADDRESS|||", str)

def mask_all(str):
    str = mask_emails(str)
    str = mask_phone(str)
    str = mask_ip(str)
    return str

In [ ]:
## Simple test. Add phone numbers.

my_str = "HELLO WORLD, my email is alexcohen@nyu.edu and another email is moo@moo.uk"
mask_emails(my_str)

In [ ]:
res = list(re.finditer(COMPILED_EMAIL_REGEX, my_str))
res[0]

In [ ]:
## Iterate through first several WARC files and show regex matches
upto = 60000
buffer = 100
for i, (entry, text) in enumerate(warc_text_iterator()):
    if i % 100 == 0: print(f"Ind: {i}")
    if i > upto: break
    lan, conf = get_lang(text)
    engl = is_english(text)
    if not engl: continue

    emails = list(re.finditer(COMPILED_EMAIL_REGEX, text))
    ips = list(re.finditer(COMPILED_IP_REGEX, text))
    phones = list(re.finditer(COMPILED_PHONE_REGEX, text))

    all_matches = phones # emails + ips + phones

    if len(all_matches) > 0: 
        print(f"Ind: {i}, uri: {entry.headers.get('WARC-Target-URI', "NONE")}, length: {entry.headers['Content-Length']}, lan: {lan}, conf: {conf}")

    for match in all_matches: 
        span = match.span()
        ind1 = max(span[0] - buffer, 0)
        ind2 = min(span[1] + buffer, len(text))
        print(text[ind1:span[0]].replace('\n', ' ').strip(), "|", text[span[0]:span[1]], "|", text[span[1]:ind2].replace('\n', ' ').strip(), '\n')
    # if not (emails == [] and ips == [] and phones == []):
        
    #     print("emails:", emails)
    #     print("ips", ips)
    #     print("phones", phones)
    #     print(text)

    ## TODO add for phone numbers and IP address

## Detect NSFW content and toxic speech
```
Use /shared-data/classifiers/dolma_fastttext_hatespeech_jigsaw_model.bin for hate and toxic speech

Use /shared-data/classifiers/dolma_fasttext_nsfw_jigsaw_model.bin for NSFW content

Use fasttext library

In [ ]:
import fasttext
nsfw_model = fasttext.load_model("../local-shared-data/classifiers/dolma_fasttext_nsfw_jigsaw_model.bin")
hatespeech_model = fasttext.load_model("../local-shared-data/classifiers/dolma_fasttext_hatespeech_jigsaw_model.bin")
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [ ]:
def get_is_nsfw(text, with_conf = False):
    stripped = text.replace('\n', ' ').strip()
    res = nsfw_model.predict(stripped)
    if with_conf:
        return res[0][0] == '__label__nsfw' , res[1][0]
    else:
        return res[0][0] == '__label__nsfw'

def get_is_hatespeech(text, with_conf = False):
    stripped = text.replace('\n', ' ').strip()
    res = hatespeech_model.predict(stripped)
    if with_conf:
        return res[0][0] == '__label__toxic' , res[1][0]
    else:
        return res[0][0] == '__label__toxic'

In [ ]:
nsfw_model.predict("this is a dog. it is my best friend. yay!")
nsfw_model.predict("sex porn fuck")
hatespeech_model.predict("i love dogs")
hatespeech_model.predict("i hate people")

In [ ]:
### Go through WARC files and show examples of NSFW and hatespeech content
## Iterate through first several WARC files and show regex matches
upto = 70000
for i, (entry, text) in enumerate(warc_text_iterator()):
    if i % 100 == 0: print(f"Ind: {i}")
    if i > upto: break
    lan, conf = get_lang(text)
    engl = is_english(text)
    if not engl: continue

    is_nsfw, conf_nsfw = get_is_nsfw(text, with_conf = True)
    is_hatespeech, conf_hatespeech = get_is_hatespeech(text, with_conf=True)
    if is_nsfw: print("IS_NSFW", conf_nsfw)
    if is_hatespeech: print("IS_HATESPEECH", conf_hatespeech)
    if is_hatespeech and not is_nsfw:
        print(lan, conf)
        print(text)
        print("\n\n")

## Implement simple Gopher quality filters 

Use nltk.word_tokenizer to count words. 

- Restrict to num words in [50, 100_000]
- Mean word length in [3, 10] chars
- Remove doc if >30% of lines end with ellipsis ...
- Remove doc is <80% of words contain at least one alphabetic character

In [ ]:
import nltk
import numpy as np

In [ ]:
## Check if document passes simple gopher quality test

HAS_ALPHANUM_RE = re.compile("[a-zA-Z]")
def has_alphanum(w):
    return re.search(HAS_ALPHANUM_RE, w) != None

def check_gopher_good(text, word_ran = (50, 100_000), mean_word_len_ran = (3, 10), max_frac_ellipsis = 0.3, min_frac_alphanum = 0.8):
    words = nltk.word_tokenize(text)

    num_words_good = len(words) >= word_ran[0] and len(words) <= word_ran[1]
    
    mean_word_len = np.mean([len(w) for w in words])
    mean_word_len_good = mean_word_len >= mean_word_len_ran[0] and mean_word_len <= mean_word_len_ran[1]

    lines = text.split('\n')
    num_lines = len(lines)
    lines_with_ellipsis = len([l for l in lines if l.strip()[-3:] == "..."])
    percent_ellipsis = lines_with_ellipsis / num_lines
    ellipsis_good = percent_ellipsis <= max_frac_ellipsis

    alphanum_words = [w for w in words if has_alphanum(w)]
    percent_alphanum = len(alphanum_words)/len(words)
    alphanum_good = percent_alphanum >= min_frac_alphanum
    # print(len(words), mean_word_len, percent_ellipsis, percent_alphanum)
    return num_words_good and mean_word_len_good and ellipsis_good and alphanum_good, (len(words), mean_word_len, percent_ellipsis, percent_alphanum)

In [ ]:
## Iterate through WARC files and show ones that pass this test

gopher_pass = ""
gopher_fail = ""

upto = 2000
for i, (entry, text) in enumerate(warc_text_iterator()):
    if i % 100 == 0: print(f"Ind: {i}")
    if i > upto: break
    lan, conf = get_lang(text)
    engl = is_english(text, 0.6)
    if not engl: continue
    if get_is_nsfw(text) or get_is_hatespeech(text): continue

    pass_gopher, reason = check_gopher_good(text)

    text = mask_all(text)

    add_str = ""
    add_str += f"lan = {lan}, conf = {conf}\n"
    add_str += f"num_words = {reason[0]}, mean_word_len = {reason[1]}, percent_ellipsis = {reason[2]}, percent_alphanum = {reason[3]}\n"
    add_str += text
    add_str += "\n\n---------\n\n"
    if pass_gopher: 
        gopher_pass += add_str
    else: 
        gopher_fail += add_str


# ## TODO change to overwrite
# with open("../data_classification_tests/gopher_pass.txt", 'w') as f:
#     # f.
#     f.write(gopher_pass)

# with open("../data_classification_tests/gopher_fail.txt", 'w') as f:
#     f.write(gopher_fail)

## Classifier of good sites
Train a quality classifier using fasttext

Will use ```/shared-data/wiki/enwiki-20260501-extracted_urls.txt.gz``` to get a list of links from wikipedia

(Actually I don't see that file in the directory so I'm not sure where it is)

Will then subsample these to get a list of urls ```subsampled_positive_urls.txt```

Will then run 
```
wget --timeout=5 \
    -i subsampled_positive_urls.txt \
    --warc-file=subsampled_positive_urls.warc \
    -O /dev/null
```

To get those webpages into a warc file

Then will use fasttext to train a classifier, with positive examples coming from subsampled_positive_urls.warc and negatvie examples coming from random samples from common crawl

Then I will set a cutoff and keep files above that cutoff





In [ ]:
## TODO

## Exact line deduplication

In [ ]:
import hashlib
from pathlib import Path
 

In [ ]:
def exact_line_deduplication(inputs_paths, output_dir):
    """
    input_paths: a list of input file paths to run deduplification on 
    output_dir: a directory to write deduplified files
    For example, if the input paths are a/1.txt and a/2.txt, and the output directory is b/, this function  writes the files b/1.txt and b/2.txt. 
    """
    print(inputs_paths)
    existing_line_hashes = set([])
    for p in inputs_paths:
        with open(p, "r") as f: 
            dedup_file = ""
            for l in f.readlines():
                hashed = hashlib.sha256(l.encode('utf-8'))
                if hashed in existing_line_hashes: continue
                else: 
                    existing_line_hashes.add(hashed)
                    dedup_file += l

        new_file = Path(output_dir) / p.name
        with open(new_file, "w") as f:
            f.write(dedup_file)


In [ ]:
documents_with_line_duplicates_paths = list(
            (Path("../tests/fixtures") / "documents_with_line_duplicates").glob("doc*.txt")
        )

exact_line_deduplication(documents_with_line_duplicates_paths, "new_files")

In [ ]:
p1.name

## Support code for minhash jaccard testing

In [4]:
import data_utils
import nltk
import numpy as np
import itertools

In [2]:
weird_text = "hElOlOl    \n to the. wéorld!!"
normalized = data_utils.normalize(weird_text)
normalized

'helolol to the weorld'

In [3]:
data_utils.make_maxhashes(normalized, 10, 2)

array([2647512596, 4069285805, 3330664876, 3714208826, 4140909709,
       2640451636, 3457303394, 3287601362, 3995006151, 2792823656])

In [3]:
doc = normalized
words = doc.split()
words

['helolol', 'to', 'the', 'weorld']

In [5]:
k = 5
mh = np.array([0 for _ in range(k)])

In [7]:
hashes = np.array([data_utils.hash_ngram(ng, i) for i in range(k)])
hashes

array([2647512596, 3332562836, 3330664876, 3714208826, 4140909709])

In [13]:
np.maximum(mh, hashes)

array([2647512596, 3332562836, 3330664876, 3714208826, 4140909709])

In [18]:
n = 2
words = doc.split()
maxhash = np.array([0 for _ in range(k)])
for ng in nltk.ngrams(words, n):
    hashes = np.array([data_utils.hash_ngram(ng, i) for i in range(k)])
    maxhash = np.maximum(maxhash, hashes)

maxhash

array([2647512596, 4069285805, 3330664876, 3714208826, 4140909709])

In [2]:
docs = [
    "hello to the world",
    "hello to the world blue",
    "my name is the bobo",
    "my name is the bobo",
    "blah blah i am alex"
]
num_docs = len(docs)
n = 2 # ngrams `n'
k = 4 # num minhash
b = 2 # num blocks 
r = k // b
jac_cutoff = 0.8
maxhashes = np.array([data_utils.make_maxhashes(d, k, n) for d in docs])
maxhashes


array([[3580284869, 1822206931, 3245192755, 4103542369],
       [3580284869, 3876272783, 3245192755, 4103542369],
       [4120100299, 2572626152, 3292383650, 4000991455],
       [4120100299, 2572626152, 3292383650, 4000991455],
       [3701710042, 3819216670, 3002209901, 3948281207]])

In [ ]:
data_utils.fuzzy_document_deduplication(x)

In [6]:
block = 0
inds = range( block * r, (block+1)*r )
distinct_inds = {}
unique_rows, inv = np.unique(maxhashes[:,inds], return_inverse = True, axis = 0)
for j in range(num_docs):
    if inv[j] in distinct_inds:
        distinct_inds[inv[j]].append(j)
    else:
        distinct_inds[inv[j]] = [j]
distinct_inds.values()

dict_values([[0], [1], [2, 3], [4]])

In [25]:
def jaccard(S1, S2):
    """compute jaccard similarity between two sets"""
    intersection = S1.intersection(S2)
    union = S1.union(S2)
    return len(intersection) / len(union)

def get_jac_similar(docs, n, jac_cutoff):
    ngram_sets = [data_utils.make_ngram_set(doc, n) for doc in docs]
    edges = []
    for i1, i2 in itertools.pairwise(range(len(ngram_sets))):
        jac = jaccard(ngram_sets[i1], ngram_sets[i2])
        if jac >= jac_cutoff: 
            edges.append((i1, i2))

    return edges

In [26]:
ind_group = [0,1,2,3]
pairs = get_jac_similar([docs[i] for i in ind_group], n, jac_cutoff)
pairs

[(2, 3)]

In [27]:
for i1, i2 in list(itertools.pairwise(range(5))):
    print(i1,i2)

0 1
1 2
2 3
3 4


In [31]:
for block in range(b):
    inds = range( block * r, (block+1)*r )
    distinct_inds = {}
    unique_rows, inv = np.unique(maxhashes[:,inds], return_inverse = True, axis = 0)
    for j in range(num_docs):
        if inv[j] in distinct_inds:
            distinct_inds[inv[j]].append(j)
        else:
            distinct_inds[inv[j]] = [j]

    for inds in distinct_inds.values():
        ## compute jaccard distance for every pair in inds
        pairs = get_jac_similar([docs[i] for i in inds], n, jac_cutoff)
        for i1, i2 in pairs:
            print(inds[i1], inds[i2])

2 3
2 3


In [10]:
def make_clusters(n, edges):
    """
    edges is a list of pairs in 0...(n-1)
    we iteratively merge any pair in edges to lead to some number of clusters
    """

    # unique index describing the connected component of each document
    # we start with each documents cluster being its index
    cluster_ind = list(range(n))

    def get_cluster_ind(i):
        if cluster_ind[i] == i:
            return i
        else:
            return get_cluster_ind(cluster_ind[i])

    # merge indices one-by-one
    # at the end of merging, cluster_ind[i] = i' where i' <= i,
    # and i' and i are in the same cluster.
    for i1, i2 in edges:
        a = get_cluster_ind(i1)
        b = get_cluster_ind(i2)
        c = min(a,b)
        d = max(a,b)
        cluster_ind[d] = c

    for j in range(n):
        cluster_ind[j] = get_cluster_ind(j)

    distinct_vals = set(cluster_ind)
    groups = {v: [] for v in distinct_vals}
    for j in range(n):
        groups[cluster_ind[j]].append(j)

    return groups.values()

In [12]:
list(make_clusters(10, [[1, 2], [4, 5], [2, 3], [2, 6]]))

[[0], [1, 2, 3, 6], [4, 5], [7], [8], [9]]

In [1]:
import data_utils
import nltk
import numpy as np
import itertools
input_paths = [f"/Users/alexandercohen/Developer/CS336-Spring-2026/assignments/assignment4-data/test_docs_jaccard/test_docs_in/d{j}.txt" for j in range(1,6)]
output_dir = "/Users/alexandercohen/Developer/CS336-Spring-2026/assignments/assignment4-data/test_docs_jaccard/test_docs_out"

k = 4
b = 2
n = 2
jac_cutoff = 0.8

In [2]:
data_utils.fuzzy_document_deduplication(input_paths, output_dir, k, b, n, jac_cutoff)

/Users/alexandercohen/Developer/CS336-Spring-2026/assignments/assignment4-data/cs336_data/data_utils.py:186: SyntaxWarning: invalid escape sequence '\c'
  The Jaccard similarity is |S(d_1) \cap S(d_2)| / |S(d_1) \cup S(d_2)|


ZeroDivisionError: division by zero